# Análise Exploratória de Exoplanetas (Fase 0)

Bem-vindo(a)! Este notebook é o nosso ponto de partida. A ideia é **explorar** um conjunto de
planetas já confirmados pela NASA para praticar Python, pandas e gráficos — e, de quebra,
entender as grandezas físicas que vão importar nas próximas fases do projeto.

Você pode executar as células **de cima para baixo**, uma por uma. Cada seção tem um texto
curto explicando *o que* estamos fazendo e *por quê*. Não precisa saber astronomia: o texto
explica os termos.

> Se ainda não baixou os dados, rode `python main.py` na raiz do projeto primeiro. Isso cria
> o arquivo `data/raw/exoplanets.csv` que usamos aqui.

In [ ]:
# Bibliotecas que vamos usar
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Permite importar as funções da pasta src/ (o notebook está dentro de notebooks/)
sys.path.append("..")
from src.preprocessing import limpar_dados, salvar_processado, COLUNAS_NUMERICAS
from src import visualization as viz

# Estilo visual limpo e opções de exibição
viz.configurar_estilo()
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

## 1. Carregamento dos dados

Vamos ler o arquivo `data/raw/exoplanets.csv`, que foi baixado do NASA Exoplanet Archive.
Cada linha é (quase) um planeta; cada coluna é uma propriedade dele.

In [ ]:
caminho_csv = Path("../data/raw/exoplanets.csv")

df = pd.read_csv(caminho_csv)
print(f"Linhas: {df.shape[0]}  |  Colunas: {df.shape[1]}")
df.head()

In [ ]:
# Tipos das colunas e quantos valores não-nulos existem em cada uma
df.info()

## 2. Dicionário das colunas

Um resumo do que cada coluna significa:

| Coluna | Significado | Unidade |
| --- | --- | --- |
| `pl_name` | Nome do planeta | texto |
| `pl_rade` | Raio do planeta | raios terrestres (1 = tamanho da Terra) |
| `pl_bmasse` | Massa do planeta | massas terrestres |
| `pl_orbper` | Período orbital (tempo de uma volta) | dias |
| `st_teff` | Temperatura da superfície da estrela | Kelvin |

## 3. Qualidade dos dados

Antes de analisar, precisamos entender os "defeitos" dos dados. Duas coisas importam aqui:

1. **Valores ausentes (`NaN`)** — nem todo planeta tem todas as medições. A massa, em
   especial, é difícil de medir e falta em muitos casos.
2. **Registros repetidos** — a tabela original guarda *uma linha por artigo científico*
   publicado sobre cada planeta. Ao baixar com `default_flag = 1` (no `data_fetch.py`) já
   pedimos só a linha "oficial" de cada planeta, mas vamos conferir mesmo assim.

In [ ]:
# Quantos valores ausentes em cada coluna, em número e em porcentagem
ausentes = (
    df.isna().sum()
    .to_frame("ausentes")
    .assign(percentual=lambda t: t["ausentes"] / len(df) * 100)
    .sort_values("percentual", ascending=False)
)
ausentes

In [ ]:
# Existem nomes de planeta repetidos?
repetidos = df["pl_name"].duplicated().sum()
print(f"Nomes de planeta repetidos: {repetidos}")

### Limpeza

Agora aplicamos a limpeza usando as funções de `src/preprocessing.py`. O que ela faz, de
forma simples:

- **Mantém um registro por planeta** (remove repetições de nome).
- **Remove linhas sem raio ou sem período**, que são as duas medições essenciais para a
  nossa análise. (Não removemos por falta de massa ou temperatura, senão perderíamos quase
  tudo.)
- **Remove valores impossíveis** (raio ou período menores ou iguais a zero).

No fim, salvamos o resultado limpo em `data/processed/` para reutilizar depois sem precisar
repetir a limpeza.

In [ ]:
df_limpo = limpar_dados(df)

print(f"Antes:  {len(df)} linhas")
print(f"Depois: {len(df_limpo)} linhas")

# Salva o dataset limpo para as próximas etapas
destino = salvar_processado(df_limpo, "../data/processed/exoplanets_limpo.csv")
print(f"Dataset limpo salvo em: {destino}")

df_limpo.head()

In [ ]:
# Estatísticas descritivas das colunas numéricas (só para ter uma ideia geral)
df_limpo[COLUNAS_NUMERICAS].describe().T

## 4. Distribuições das variáveis

Vamos ver como os valores se espalham. Dados astronômicos costumam ter valores MUITO
diferentes entre si (de planetas menores que a Terra a gigantes), então além do histograma
normal usamos também a **escala logarítmica** (`log10`), que "aproxima" as ordens de
grandeza e deixa o gráfico mais legível.

In [ ]:
# Histogramas em escala normal
viz.plot_distribuicoes(df_limpo, COLUNAS_NUMERICAS, bins=40, log=False)
plt.show()

In [ ]:
# Os mesmos dados em escala logarítmica (mais fácil de ler)
viz.plot_distribuicoes(df_limpo, ["pl_rade", "pl_bmasse", "pl_orbper"], bins=40, log=True)
plt.show()

## 5. Relações entre variáveis

Aqui olhamos se as variáveis "andam juntas". Por exemplo: planetas maiores tendem a ser mais
massivos? A cor dos pontos mostra a temperatura da estrela. Usamos escala log nos dois eixos
para caber planetas pequenos e gigantes no mesmo gráfico.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df_limpo, x="pl_rade", y="pl_bmasse", hue="st_teff",
                palette="viridis", alpha=0.65, ax=axes[0])
axes[0].set(xscale="log", yscale="log", title="Massa x Raio",
            xlabel="Raio (raios terrestres)", ylabel="Massa (massas terrestres)")

sns.scatterplot(data=df_limpo, x="pl_orbper", y="pl_rade", hue="st_teff",
                palette="magma", alpha=0.65, ax=axes[1])
axes[1].set(xscale="log", yscale="log", title="Raio x Período orbital",
            xlabel="Período orbital (dias)", ylabel="Raio (raios terrestres)")

plt.tight_layout()
plt.show()

## 6. Classificação simples por tamanho

Uma forma fácil de resumir os planetas é agrupá-los por faixa de raio. Isto é apenas uma
classificação aproximada e didática (não é uma definição oficial).

In [ ]:
faixas = [0, 1.25, 2, 6, 15, np.inf]
rotulos = ["Rochoso/Terrestre", "Super-Terra", "Sub-Netuno/Netuno", "Gigante gasoso", "Muito grande"]

df_limpo = df_limpo.copy()
df_limpo["classe_tamanho"] = pd.cut(df_limpo["pl_rade"], bins=faixas, labels=rotulos)

contagem = df_limpo["classe_tamanho"].value_counts().reindex(rotulos)

plt.figure(figsize=(10, 5))
sns.barplot(x=contagem.index, y=contagem.values, color="#4C78A8")
plt.title("Quantidade de planetas por classe de tamanho")
plt.xlabel("Classe (aproximada)")
plt.ylabel("Quantidade")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

contagem

## 7. Triagem exploratória de planetas interessantes

Como curiosidade, vamos filtrar planetas com características parecidas com as da Terra:

- raio próximo ao terrestre (entre 0,8 e 1,8),
- estrela com temperatura parecida com a do Sol (entre 4500 e 6500 K),
- período orbital intermediário (entre 50 e 500 dias).

⚠️ **Atenção:** isto é só uma triagem para levantar curiosidade — **não** é uma conclusão
sobre habitabilidade, que exigiria muito mais informação (luminosidade da estrela, distância
orbital, atmosfera, etc.).

In [ ]:
interessantes = df_limpo[
    df_limpo["pl_rade"].between(0.8, 1.8)
    & df_limpo["st_teff"].between(4500, 6500)
    & df_limpo["pl_orbper"].between(50, 500)
].copy()

print(f"Planetas na triagem: {len(interessantes)}")
interessantes.sort_values("pl_rade")[["pl_name", "pl_rade", "pl_orbper", "st_teff"]].head(15)

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_limpo, x="pl_orbper", y="pl_rade",
                color="lightgray", alpha=0.35, label="Demais planetas")
sns.scatterplot(data=interessantes, x="pl_orbper", y="pl_rade",
                color="#D62728", s=70, label="Triagem de interesse")
plt.xscale("log")
plt.yscale("log")
plt.title("Planetas destacados pela triagem")
plt.xlabel("Período orbital (dias)")
plt.ylabel("Raio (raios terrestres)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Conclusões e próximos passos

O que aprendemos nesta primeira análise:

- A tabela tem muitas medições de **raio** e **período**, mas a **massa** falta na maioria
  dos planetas.
- As distribuições são bem **assimétricas** e cheias de valores extremos — por isso a escala
  logarítmica ajuda tanto.
- Existe uma relação clara entre **massa e raio** (planetas maiores tendem a ser mais massivos).
- Nossa triagem "tipo Terra" é só um aperitivo: analisar habitabilidade de verdade exige mais
  variáveis.

**Próximo passo (quando o grupo estiver confortável):** partir para a Fase 1 do
[roadmap](../docs/roadmap.md) — baixar a *curva de luz* de uma estrela conhecida (Kepler-10)
e ver o trânsito do planeta com nossos próprios olhos.